In [ ]:
!pip install --upgrade transformers accelerate bitsandbytes trl datasets peft tokenizers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Unin

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig
from trl import SFTTrainer

In [ ]:
from huggingface_hub import login
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN") # Fetching the Hugging Face token from Colab secrets
login(token = hf_token) # Logging into Hugging Face Hub to access models and other resources

In [ ]:
base_model = 'meta-llama/Llama-3.2-3B-Instruct'

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
# Configure the tokenizer to apply padding on the right side of the input
# This is often the default for causal language models to ensure alignment during training or inference
tokenizer.padding_side = "right"

In [ ]:
dataset = load_dataset("Victorano/customer-support-1k", split="train")
dataset = dataset.remove_columns(['flags', 'category','intent','text'])
dataset = dataset.train_test_split(test_size=0.2)

README.md:   0%|          | 0.00/471 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/648k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response'],
        num_rows: 800
    })
    test: Dataset({
        features: ['instruction', 'response'],
        num_rows: 200
    })
})

In [ ]:
instruction = """You are a helpful and efficient customer support bot designed to assist users by providing answers to frequently asked questions (FAQs) related to our products and services. Your responses should be concise, clear, and friendly, ensuring the user feels heard and supported. If the user’s question is outside the scope of the FAQ, gently direct them to contact customer support.

Always prioritize accuracy and clarity in your answers.
If the user asks a complex question, break it down into smaller, manageable parts and answer step-by-step.
Provide useful links or references to detailed documentation when appropriate.
Use a friendly and professional tone, ensuring the response is easy to understand.
If the FAQ does not cover the question, offer an apology and suggest contacting customer support.
"""

def template(row):
  row_json = [{"role":"system", "content":instruction},
              {"role":"user","content":row["instruction"]},
              {"role":"assistant","content":row["response"]}]
  row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)

  return row

dataset = dataset.map(template, num_proc=4)

Map (num_proc=4):   0%|          | 0/800 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
dataset['train']['text'][10]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 21 Apr 2026\n\nYou are a helpful and efficient customer support bot designed to assist users by providing answers to frequently asked questions (FAQs) related to our products and services. Your responses should be concise, clear, and friendly, ensuring the user feels heard and supported. If the user’s question is outside the scope of the FAQ, gently direct them to contact customer support.\n\nAlways prioritize accuracy and clarity in your answers.\nIf the user asks a complex question, break it down into smaller, manageable parts and answer step-by-step.\nProvide useful links or references to detailed documentation when appropriate.\nUse a friendly and professional tone, ensuring the response is easy to understand.\nIf the FAQ does not cover the question, offer an apology and suggest contacting customer support.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nassistanc

In [ ]:
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.2,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


trainable params: 1,146,880 || all params: 3,213,896,704 || trainable%: 0.0357


In [ ]:
training_arguments = TrainingArguments(
    output_dir="./results",  # Directory where the results will be saved
    num_train_epochs=1,  # Number of training epochs
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    warmup_steps=5,  # Number of warmup steps to gradually increase the learning rate during training
    learning_rate=2e-4,  # Learning rate for the optimizer
    bf16=True,  # Enabling BFloat16 floating point precision for faster training on GPUs that support it (reduces memory usage) and better numerical stability
    report_to="none",  # Disabling logging/reporting to external services (e.g., TensorBoard, Weights & Biases)
)

# Initializing the SFTTrainer for supervised fine-tuning
trainer = SFTTrainer(
    model=model,  # The pre-trained model to be fine-tuned
    train_dataset=dataset["train"], # The dataset used for training
    eval_dataset=dataset["test"],  # The dataset used for validation
    processing_class=tokenizer,  # Tokenizer to process input text for the model
    args=training_arguments,  # The training arguments defined above
)

In [ ]:
model.train()
trainer.train()

In [28]:
def generate(input_prompt):
  message = [
      {"role":"system", "content":instruction},
      {"role":"user", "content":input_prompt}
  ]

  prompt = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)
  # Move input tensors to the same device as the model (CUDA)
  inputs = {k: v.to(model.device) for k, v in inputs.items()}
  outputs = model.generate(**inputs, max_new_tokens=2048, num_return_sequences=1)
  text = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return text

In [29]:
response = generate("What is the return policy?")
print(response)

system

Cutting Knowledge Date: December 2023
Today Date: 21 Apr 2026

You are a helpful and efficient customer support bot designed to assist users by providing answers to frequently asked questions (FAQs) related to our products and services. Your responses should be concise, clear, and friendly, ensuring the user feels heard and supported. If the user’s question is outside the scope of the FAQ, gently direct them to contact customer support.

Always prioritize accuracy and clarity in your answers.
If the user asks a complex question, break it down into smaller, manageable parts and answer step-by-step.
Provide useful links or references to detailed documentation when appropriate.
Use a friendly and professional tone, ensuring the response is easy to understand.
If the FAQ does not cover the question, offer an apology and suggest contacting customer support.user

What is the return policy?assistant

I'm here to help! I understand that you're interested in knowing about our return pol

In [30]:
print(response.split("assistant")[-1])



I'm here to help! I understand that you're interested in knowing about our return policy. Our return policy is designed to provide you with flexibility and satisfaction. You can return or exchange an item within {{Time Frame}} days from the date of delivery. To initiate the return process, please contact our customer support team at {{Customer Support Phone Number}} or {{Customer Support Email}}. They will guide you through the steps and ensure a smooth return.


In [31]:
model.save_pretrained("/content/customer-faq-llama-3.2-3B") # Saves the model under the same directory.

In [34]:
# To push the model to hugginface
# The error indicates that the token used does not have the rights to create a model
# under the namespace 'iharne03'.
# Please ensure your Hugging Face token (HF_TOKEN) has 'write' permissions.
# If 'iharne03' is not your Hugging Face username, you need to change the repository ID
# to 'YOUR_HF_USERNAME/customer-faq-llama-3.2-3B' where 'YOUR_HF_USERNAME' is your actual username.
model.push_to_hub("iharne03/customer-faq-llama-3.2-3B")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  27%|##6       |  621kB / 2.31MB            

CommitInfo(commit_url='https://huggingface.co/iharne03/customer-faq-llama-3.2-3B/commit/9d9cb259682e7a0ba3b05b980d726f738fd0c90b', commit_message='Upload model', commit_description='', oid='9d9cb259682e7a0ba3b05b980d726f738fd0c90b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/iharne03/customer-faq-llama-3.2-3B', endpoint='https://huggingface.co', repo_type='model', repo_id='iharne03/customer-faq-llama-3.2-3B'), pr_revision=None, pr_num=None)